In [1]:
import random

def get_conflicts(state):
    """Calculate the number of diagonal conflicts in the current state."""
    n = len(state)
    conflicts = 0
    for i in range(n):
        for j in range(i + 1, n):
            # Since we use a permutation, row/col conflicts are impossible.
            # We only need to check diagonal conflicts: |row_i - row_j| == |col_i - col_j|
            if abs(i - j) == abs(state[i] - state[j]):
                conflicts += 1
    return conflicts

def get_best_swap_neighbor(state):
    """Generate all possible swaps of two elements and find the best neighbor."""
    n = len(state)
    current_conflicts = get_conflicts(state)
    best_neighbor = list(state)
    best_conflicts = current_conflicts

    # Generate all pairs to swap
    for i in range(n):
        for j in range(i + 1, n):
            neighbor = list(state)
            # Swap columns
            neighbor[i], neighbor[j] = neighbor[j], neighbor[i]

            neighbor_conflicts = get_conflicts(neighbor)
            if neighbor_conflicts < best_conflicts:
                best_conflicts = neighbor_conflicts
                best_neighbor = neighbor

    return best_neighbor, best_conflicts

In [13]:
def print_board(state):
    n = len(state)
    for row in range(n):
        line = []
        for col in range(n):
            if state[col] == row:
                line.append("Q")
            else:
                line.append(".")
        print(" ".join(line))

def hill_climbing_with_trace(initial_state):
    state = list(initial_state)
    n = len(state)
    step = 0
    restart_count = 0
    current_conflicts = get_conflicts(state)

    print(f"Step {step} (Initial): State = {state}, Conflicts = {current_conflicts}")
    print_board(state)
    print("-" * 40)

    while True:
        if current_conflicts == 0:
            print(f"Success! 0 conflicts reached after {restart_count} restart(s).")
            break

        neighbor, neighbor_conflicts = get_best_swap_neighbor(state)

        if neighbor_conflicts >= current_conflicts:
            restart_count += 1
            print(f"Stuck in local minimum at State = {state} (Conflicts = {current_conflicts}).")
            print(f"--- Restarting search with a random state (Restart #{restart_count}) ---")
            # Generate a random permutation of columns to escape the local minimum
            state = list(range(n))
            random.shuffle(state)
            current_conflicts = get_conflicts(state)
            step += 1
            print(f"Step {step} (New Start): State = {state}, Conflicts = {current_conflicts}")
            print_board(state)
            print("-" * 40)
            continue

        # Identify which indices were swapped for tracing
        swapped_indices = [i for i in range(n) if state[i] != neighbor[i]]

        state = neighbor
        current_conflicts = neighbor_conflicts
        step += 1

        print(f"Step {step}: Swapped indices {swapped_indices} -> State = {state}, Conflicts = {current_conflicts}")
        print_board(state)
        print("-" * 40)

# Run the trace with random restarts for 6 queens
hill_climbing_with_trace([3, 1, 2, 0])

Step 0 (Initial): State = [3, 1, 2, 0], Conflicts = 2
. . . Q
. Q . .
. . Q .
Q . . .
----------------------------------------
Step 1: Swapped indices [0, 1] -> State = [1, 3, 2, 0], Conflicts = 1
. . . Q
Q . . .
. . Q .
. Q . .
----------------------------------------
Step 2: Swapped indices [2, 3] -> State = [1, 3, 0, 2], Conflicts = 0
. . Q .
Q . . .
. . . Q
. Q . .
----------------------------------------
Success! 0 conflicts reached after 0 restart(s).


Hill Climbing is not suitable for finding all the soltions, it gets stops as soon as it finds the first soltion. If we want all the solutions then the best approach is Backtracking Depth First Search(DFS)

In [5]:
def solve_n_queens_backtracking(n):
    solutions = []

    def is_valid(state, col, row):
        # Check diagonal conflicts with previously placed queens
        for prev_col in range(col):
            prev_row = state[prev_col]
            if abs(prev_col - col) == abs(prev_row - row):
                return False
        return True

    def backtrack(col, current_state, remaining_rows):
        if col == n:
            solutions.append(list(current_state))
            return

        # Try placing a queen in each available row for the current column
        for row in list(remaining_rows):
            if is_valid(current_state, col, row):
                current_state.append(row)
                remaining_rows.remove(row)

                backtrack(col + 1, current_state, remaining_rows)

                # Backtrack
                remaining_rows.add(row)
                current_state.pop()

    backtrack(0, [], set(range(n)))
    return solutions

# Find and display all solutions for N=4
solutions = solve_n_queens_backtracking(4)
print(f"Found {len(solutions)} solutions for 4-Queens:")
for idx, sol in enumerate(solutions, 1):
    print(f"\nSolution {idx}: State = {sol}")
    print_board(sol)
    print("-" * 20)

Found 2 solutions for 4-Queens:

Solution 1: State = [1, 3, 0, 2]
. . Q .
Q . . .
. . . Q
. Q . .
--------------------

Solution 2: State = [2, 0, 3, 1]
. Q . .
. . . Q
Q . . .
. . Q .
--------------------
